# Project: Semantic Search with Transformers

In [1]:
# Run this cell beforehand so you do not see any warnings
import warnings
warnings.filterwarnings('ignore')

## Task 1: Import the Libraries

In [1]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn import preprocessing
import faiss
import numpy as np
import pickle

/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Task 2: Load the Data

In [2]:
# Load the dataset from a JSON file into a Pandas DataFrame  
data = pd.read_json('/usercode/arxivData.json')  

# Remove unnecessary columns
df = data.drop(columns=["author", "link", 'tag'])  

# Print the total number of unique Machine Learning papers  
print("Number of Machine Learning papers: ", df.id.unique().shape[0])  

# Display the first few rows of the modified DataFrame  
df.head() 

Number of Machine Learning papers:  41000


,day,id,month,summary,title,year
0,1,1802.00209v1,2,We propose an architecture for VQA which utili...,Dual Recurrent Attention Units for Visual Ques...,2018
1,12,1603.03827v1,3,Recent approaches based on artificial neural n...,Sequential Short-Text Classification with Recu...,2016
2,2,1606.00776v2,6,We introduce the multiresolution recurrent neu...,Multiresolution Recurrent Neural Networks: An ...,2016
3,23,1705.08142v2,5,Multi-task learning is motivated by the observ...,Learning what to share between loosely related...,2017
4,7,1709.02349v2,9,We present MILABOT: a deep reinforcement learn...,A Deep Reinforcement Learning Chatbot,2017


In [4]:
# Load the pre-trained SentenceTransformer model
model = SentenceTransformer('distilbert-base-nli-stsb-mean-tokens')

# Check if a GPU (CUDA) is available and move the model to GPU if possible
if torch.cuda.is_available():
    model = model.to(torch.device("cuda"))

# Print the device the model is running on (CPU or GPU)
print(model.device)

cpu


## Task 3: Retrieve the Model

In [ ]:
embeddings = model.encode(df.summary.to_list()[:2000], show_progress_bar=True)
with open('/usercode/new_embeddings.pickle', 'wb') as pkl:
  pickle.dump(embeddings, pkl)

## Task 4: Generate or Load the Embeddings

In [5]:
embeddings = model.encode(df.summary.to_list()[:2000], show_progress_bar=True)
with open('/usercode/new_embeddings.pickle', 'wb') as pkl:
  pickle.dump(embeddings, pkl)

Batches: 100%|██████████| 63/63 [07:47<00:00,  7.42s/it]


In [3]:
with open('/usercode/default_embeddings.pickle', 'rb') as pkl:
  embeddings = pickle.load(pkl)

In [6]:
with open('/usercode/new_embeddings.pickle', 'rb') as pkl:
  embeddings = pickle.load(pkl)


length = len(embeddings)
print ('length: ', length)
print('Shape of the one embedding: ', embeddings[0].shape)
print("Number of embeddings:", len(embeddings))
print("Dimension of each embedding:", embeddings[0].shape)
print("Total values stored:", len(embeddings) * embeddings[0].shape[0])

length:  2000
Shape of the one embedding:  (768,)
Number of embeddings: 2000
Dimension of each embedding: (768,)
Total values stored: 1536000


## Task 5: Data Preparation and Helper Methods

In [9]:
le = preprocessing.LabelEncoder()
df['id'] = le.fit_transform(df['id'])
df.head(2)

def id2info(df, I, column):
    return [list(df[df.id == idx][column]) for idx in I]

,day,id,month,summary,title,year
0,1,36693,2,We propose an architecture for VQA which utili...,Dual Recurrent Attention Units for Visual Ques...,2018
1,12,18198,3,Recent approaches based on artificial neural n...,Sequential Short-Text Classification with Recu...,2016


## Task 6: Set up the Index

In [10]:
embeddings = np.array(embeddings).astype("float32")
print (embeddings.shape[1])
index = faiss.IndexFlatL2(embeddings.shape[1])
index = faiss.IndexIDMap(index)
index.add_with_ids(embeddings, df['id'][:length])

print("Number of embeddings in the Faiss index: ", index.ntotal)

768
Number of embeddings in the Faiss index:  2000


## Task 7: Search with a Summary

In [12]:
df.iloc[1337, [3, 1]]

D, I = index.search(np.array([embeddings[1337]]), k=10)
pd.DataFrame({'L2 distance': D.flatten().tolist(), 'ML paper IDs': I.flatten().tolist(), 'ML paper titles': id2info(df, I.flatten(), 'title'), 'Summaries': id2info(df, I.flatten(), 'summary')}).head(10)

,L2 distance,ML paper IDs,ML paper titles,Summaries
0,0.000000,12964,[Convolutional Neural Networks for joint objec...,[In this paper we study the application of con...
1,61.530529,11503,[Deep Metric Learning for Practical Person Re-...,[Various hand-crafted features and metric lear...
2,65.805099,11377,[Cortical spatio-temporal dimensionality reduc...,"[The visual systems of many mammals, including..."
3,67.032898,11142,[Heterogeneous Multi-task Learning for Human P...,[We propose an heterogeneous multi-task learni...
4,69.937653,30623,[Neural Expectation Maximization],[Many real world tasks such as reasoning and p...
5,70.530083,13876,[Pixel-wise Deep Learning for Contour Detection],[We address the problem of contour detection v...
6,73.332405,18371,[Sparse Activity and Sparse Connectivity in Su...,[Sparseness is a useful regularizer for learni...
7,73.559280,10182,[Deeply Coupled Auto-encoder Networks for Cros...,[The comparison of heterogeneous samples exten...
8,73.715988,21064,[Crafting a multi-task CNN for viewpoint estim...,[Convolutional Neural Networks (CNNs) were rec...
9,73.727539,18667,[Deep Aesthetic Quality Assessment with Semant...,[Human beings often assess the aesthetic quali...


## Task 8: Search with a Prompt


In [13]:
user_query = "The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data"
embed = model.encode(list(user_query))
D, I = index.search(np.array([embed]).squeeze().astype("float32"), k=10)

results = {'L2 distances':D.flatten().tolist(), 'ML paper IDs':I.flatten().tolist(), "Titles": id2info(df, I.flatten(), 'title'), "Summaries": id2info(df, I.flatten(), 'summary')}


In [14]:
pd.DataFrame(results).head(10)

,L2 distances,ML paper IDs,Titles,Summaries
0,398.835114,26890,[Abstract Syntax Networks for Code Generation ...,[Tasks like code generation and semantic parsi...
1,401.845093,30018,[Dual Rectified Linear Units (DReLUs): A Repla...,"[In this paper, we introduce a novel type of R..."
2,403.243622,7184,[KSU KDD: Word Sense Induction by Clustering i...,[We describe our language-independent unsuperv...
3,403.410858,28659,[Topic supervised non-negative matrix factoriz...,[Topic models have been extensively used to or...
4,404.912506,34786,[Don't Just Assume; Look and Answer: Overcomin...,[A number of studies have found that today's V...
5,406.426178,30468,[Regularizing and Optimizing LSTM Language Mod...,"[Recurrent neural networks (RNNs), such as lon..."
6,410.384216,16734,[Learning the Dimensionality of Word Embeddings],[We describe a method for learning word embedd...
7,410.741394,12050,[HD-CNN: Hierarchical Deep Convolutional Neura...,"[In image classification, visual separability ..."
8,414.264343,11912,[Taking into Account the Differences between A...,[Actively sampled data can have very different...
9,414.365906,31758,[Self-Guiding Multimodal LSTM - when we do not...,"[In this paper, a self-guiding multimodal LSTM..."


# End